<a href="https://colab.research.google.com/github/Mohammed-Ishaq2004/160923733005-MohammedIshaq/blob/main/practise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#12th
import tensorflow as tf
import numpy as np

# Load data
(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255
x_train = x_train.reshape(-1, 28, 28, 1)

# Generator
G = tf.keras.Sequential([
    tf.keras.layers.Input((100,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(784, activation='sigmoid'),
    tf.keras.layers.Reshape((28, 28, 1))
])

# Discriminator
D = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

D.compile(optimizer='adam', loss='binary_crossentropy')

# Combined GAN
D.trainable = False
gan = tf.keras.Sequential([G, D])
gan.compile(optimizer='adam', loss='binary_crossentropy')

# Training
for i in range(5):
    idx = np.random.randint(0, len(x_train), 64)
    real = x_train[idx]

    noise = np.random.normal(size=(64, 100))
    fake = G.predict(noise, verbose=0)

    D.trainable = True
    D.train_on_batch(real, np.ones((64, 1)))
    D.train_on_batch(fake, np.zeros((64, 1)))

    D.trainable = False
    noise = np.random.normal(size=(64, 100))
    loss = gan.train_on_batch(noise, np.ones((64, 1)))

    print("Epoch:", i+1, "Generator loss:", loss)

# Generate images
images = G.predict(np.random.normal(size=(10, 100)), verbose=0)
print("Generated images shape:", images.shape)


Epoch: 1 Generator loss: 0.27345657
Epoch: 2 Generator loss: 0.9772679
Epoch: 3 Generator loss: 1.8188443
Epoch: 4 Generator loss: 2.5373807
Epoch: 5 Generator loss: 3.0886545
Generated images shape: (10, 28, 28, 1)


In [7]:
# 11th
import tensorflow as tf
import numpy as np

# Load data
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train[..., None].astype('float32') / 255
x_test = x_test[..., None].astype('float32') / 255

# Add noise
x_train_noisy = np.clip(x_train + 0.5 * np.random.normal(size=x_train.shape), 0, 1)
x_test_noisy = np.clip(x_test + 0.5 * np.random.normal(size=x_test.shape), 0, 1)

# Autoencoder
model = tf.keras.Sequential([
    tf.keras.layers.Input((28, 28, 1)),
    tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    tf.keras.layers.UpSampling2D(2),
    tf.keras.layers.Conv2D(1, 3, activation='sigmoid', padding='same')
])

model.compile(optimizer='adam', loss='binary_crossentropy')

# Added batch_size=128 and reduced epochs to 3 for a faster run
model.fit(
    x_train_noisy, x_train,
    epochs=3,               # Reduced from 5 to 3
    batch_size=128,         # Added batch size to speed up steps per epoch
    validation_data=(x_test_noisy, x_test)
)

# Test
loss = model.evaluate(x_test_noisy, x_test, verbose=0)
print("Test loss:", round(loss, 4))

decoded = model.predict(x_test_noisy[:3], verbose=0)
print("Denoised images:", decoded.shape)

Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 61s 127ms/step - loss: 0.1916 - val_loss: 0.1164
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 58s 123ms/step - loss: 0.1143 - val_loss: 0.1112
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 58s 123ms/step - loss: 0.1112 - val_loss: 0.1093
Test loss: 0.1093
Denoised images: (3, 28, 28, 1)


In [5]:
#8,9,10
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

NUM_WORDS = 10000
MAXLEN = 500

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=NUM_WORDS)

x_train = pad_sequences(x_train, maxlen=MAXLEN)
x_test = pad_sequences(x_test, maxlen=MAXLEN)

# Change this line for each experiment
RNN_TYPE = SimpleRNN

model = Sequential([
    Embedding(NUM_WORDS, 128),
    RNN_TYPE(128),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

model.fit(
    x_train, y_train,
    epochs=2,
    batch_size=128,
    validation_split=0.2
)

_, acc = model.evaluate(x_test, y_test, verbose=0)
print("Test accuracy:", acc)

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 132s 833ms/step - accuracy: 0.6055 - loss: 0.6463 - val_accuracy: 0.6618 - val_loss: 0.6022
Epoch 2/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 140s 816ms/step - accuracy: 0.8140 - loss: 0.4128 - val_accuracy: 0.7522 - val_loss: 0.5048
Test accuracy: 0.7615600228309631


In [6]:
#7th
import tensorflow as tf

L = tf.keras.layers

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Preprocessing
x_train = x_train[..., None].astype('float32') / 255.0
x_test = x_test[..., None].astype('float32') / 255.0

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# CHANGE ONLY THIS LINE
ARCH = "LeNet"       # "AlexNet" or "VGG"

# LeNet-5
if ARCH == "LeNet":
    model = tf.keras.Sequential([
        L.Input((28, 28, 1)),
        L.Conv2D(6, 5, activation='tanh'),
        L.AveragePooling2D(2),
        L.Conv2D(16, 5, activation='tanh'),
        L.AveragePooling2D(2),
        L.Flatten(),
        L.Dense(120, activation='tanh'),
        L.Dense(84, activation='tanh'),
        L.Dense(10, activation='softmax')
    ], name="LeNet-5")

# AlexNet
elif ARCH == "AlexNet":
    model = tf.keras.Sequential([
        L.Input((28, 28, 1)),
        L.Conv2D(32, 3, activation='relu', padding='same'),
        L.MaxPooling2D(),
        L.Conv2D(64, 3, activation='relu', padding='same'),
        L.MaxPooling2D(),
        L.Conv2D(128, 3, activation='relu', padding='same'),
        L.Conv2D(128, 3, activation='relu', padding='same'),
        L.Conv2D(64, 3, activation='relu', padding='same'),
        L.MaxPooling2D(),
        L.Flatten(),
        L.Dense(512, activation='relu'),
        L.Dropout(0.5),
        L.Dense(256, activation='relu'),
        L.Dropout(0.5),
        L.Dense(10, activation='softmax')
    ], name="AlexNet")

# VGG
else:
    model = tf.keras.Sequential([
        L.Input((28, 28, 1)),
        L.Conv2D(64, 3, activation='relu', padding='same'),
        L.Conv2D(64, 3, activation='relu', padding='same'),
        L.MaxPooling2D(),
        L.Conv2D(128, 3, activation='relu', padding='same'),
        L.Conv2D(128, 3, activation='relu', padding='same'),
        L.MaxPooling2D(),
        L.Conv2D(256, 3, activation='relu', padding='same'),
        L.Conv2D(256, 3, activation='relu', padding='same'),
        L.MaxPooling2D(),
        L.Flatten(),
        L.Dense(512, activation='relu'),
        L.Dropout(0.5),
        L.Dense(256, activation='relu'),
        L.Dropout(0.5),
        L.Dense(10, activation='softmax')
    ], name="VGG")

# Compile model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Architecture:", ARCH)
print("Parameters:", model.count_params())

# Train model
model.fit(x_train, y_train, epochs=5,
          batch_size=32,
          validation_data=(x_test, y_test))

# Evaluate
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print("Test accuracy:", round(acc, 4))

Architecture: LeNet
Parameters: 44426
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 33s 17ms/step - accuracy: 0.9287 - loss: 0.2360 - val_accuracy: 0.9667 - val_loss: 0.1055
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 32s 17ms/step - accuracy: 0.9721 - loss: 0.0887 - val_accuracy: 0.9780 - val_loss: 0.0665
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - accuracy: 0.9806 - loss: 0.0608 - val_accuracy: 0.9814 - val_loss: 0.0588
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - accuracy: 0.9854 - loss: 0.0464 - val_accuracy: 0.9808 - val_loss: 0.0600
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 34s 18ms/step - accuracy: 0.9879 - loss: 0.0374 - val_accuracy: 0.9853 - val_loss: 0.0455
Test accuracy: 0.9853
